# Snap Distance Append

## Centroid-to-Network-Node Distance for all_access

**Tess Vu**

This notebook appends two columns to the existing `tess_all_access.csv`:

- `walk_snap_dist_m` are meters from each SAL centroid to its nearest walk-network node.
- `drive_snap_dist_m` are meters from each SAL centroid to its nearest drive-network node.

No 2SFCA or Dijkstra computation is re-run.
The bottleneck is loading the four `.graphml` files from disk (~10–20 min total).
The snap computation itself (KDTree nearest-node + Euclidean distance) is instant.

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import os
import warnings

warnings.filterwarnings("ignore")

In [ ]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()          # repository root; notebooks run from notebooks/
DATA = ROOT / "data"
RAW = DATA / "raw"
INTERMEDIATE = DATA / "intermediate"
FINAL = DATA / "final"


IN_ALL_ACCESS = INTERMEDIATE / "all_access.csv"
IN_SAL_DEDUP = INTERMEDIATE / "sal_w_ward_dedup" / "sal_w_ward_dedup.shp"
IN_NETWORK_DIR = INTERMEDIATE / "networks"
IN_POP_PRED = INTERMEDIATE / "pop_pred_final.csv"
OUT_ALL_ACCESS_SNAP = INTERMEDIATE / "all_access_w_snap.csv"


## CONFIGURATION

In [ ]:
# PROJECTED CRS MATCHING THE 2SFCA NOTEBOOK.
PROJECTED_CRS = "EPSG:32735"

# GRAPH PATHS KEYED AS "Province_mode" TO MATCH all_access PR_NAME VALUES.
GRAPHML_PATHS = {
    "Gauteng_walk": IN_NETWORK_DIR / "network_gauteng_walk.graphml",
    "Gauteng_drive": IN_NETWORK_DIR / "network_gauteng_drive.graphml",
    "KwaZulu-Natal_walk": IN_NETWORK_DIR / "network_kwazulu_natal_walk.graphml",
    "KwaZulu-Natal_drive": IN_NETWORK_DIR / "network_kwazulu_natal_drive.graphml",
}

print(f"ALL_ACCESS INPUT: {IN_ALL_ACCESS}")
print(f"SAL SHAPEFILE: {IN_SAL_DEDUP}")
print(f"PROJECTED CRS: {PROJECTED_CRS}")


## LOAD ALL_ACCESS AND COMPUTE SAL CENTROIDS

In [ ]:
# Load the existing all_access CSV.
all_access = pd.read_csv(IN_ALL_ACCESS, low_memory = False)
all_access["EA_CODE"] = pd.to_numeric(all_access["EA_CODE"], errors = "coerce").astype("Int64")

print(f"ALL_ACCESS LOADED: {len(all_access)} rows")
print(f"Columns with snap distance already: {[c for c in all_access.columns if 'snap' in c]}")

# Load SAL polygon shapefile and compute centroids in projected CRS.
# The centroid computation here matches exactly what the 2SFCA notebook does:
# sal.set_geometry(sal.geometry.centroid) after to_crs(PROJECTED_CRS).
sal_geo = gpd.read_file(IN_SAL_DEDUP)
sal_geo["EA_CODE"] = pd.to_numeric(sal_geo["EA_CODE"], errors = "coerce").astype("Int64")

sal_proj = sal_geo.to_crs(PROJECTED_CRS)
sal_proj["centroid_x"] = sal_proj.geometry.centroid.x
sal_proj["centroid_y"] = sal_proj.geometry.centroid.y

print(f"SAL CENTROIDS COMPUTED: {len(sal_proj)} features")
print(f"CRS: {sal_proj.crs}")

In [ ]:
# Build a lookup table from EA_CODE to centroid coordinates.
centroid_lookup = (
    sal_proj[["EA_CODE", "centroid_x", "centroid_y"]]
    .drop_duplicates(subset = "EA_CODE")
    .set_index("EA_CODE")
)

# Merge centroid coordinates onto all_access.
# Used only during this notebook as helper columns; dropped before saving.
all_access = all_access.merge(
    centroid_lookup,
    on = "EA_CODE",
    how = "left"
)

matched = all_access["centroid_x"].notna().sum()
print(f"CENTROID MERGE: {matched}/{len(all_access)} SALs matched")

## COMPUTE SNAP DISTANCES PER PROVINCE AND NETWORK MODE

Each graph is loaded once, projected, and discarded.
`ox.nearest_nodes()` uses an internal KDTree — no routing is performed.
Snap distance is the straight-line Euclidean distance in meters from the SAL
centroid to the node it snapped to, computed in the projected CRS.

In [ ]:
# Initialize output columns with NaN.
all_access["walk_snap_dist_m"] = np.nan
all_access["drive_snap_dist_m"] = np.nan

for graph_key, graphml_path in GRAPHML_PATHS.items():
    # Parse province name and mode from the key.
    province, mode = graph_key.rsplit("_", 1)
    col_name = f"{mode}_snap_dist_m"

    print(f"LOADING GRAPH: {graph_key}")

    # Load raw graph from disk, then project into metric CRS.
    G_raw = ox.load_graphml(graphml_path)
    G = ox.project_graph(G_raw, to_crs = PROJECTED_CRS)

    print(f"  Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")

    # Extract node coordinates from the projected graph into a lookup DataFrame.
    # The projected graph stores node x and y in the metric CRS.
    node_coord_dict = {
        node_id: (data["x"], data["y"])
        for node_id, data in G.nodes(data = True)
    }
    node_x_arr = np.array([node_coord_dict[n][0] for n in node_coord_dict])
    node_y_arr = np.array([node_coord_dict[n][1] for n in node_coord_dict])
    node_id_arr = np.array(list(node_coord_dict.keys()))

    # Subset all_access to this province.
    province_mask = all_access["PR_NAME"] == province
    prov_df = all_access.loc[province_mask].copy()

    print(f"  SALs in {province}: {len(prov_df)}")

    # Drop any rows with missing centroid coordinates (uninhabited SALs with no geometry).
    valid_mask = prov_df["centroid_x"].notna() & prov_df["centroid_y"].notna()
    prov_valid = prov_df.loc[valid_mask]

    # Snap each SAL centroid to the nearest network node.
    # ox.nearest_nodes returns node IDs in the same order as the input coordinate arrays.
    snapped_node_ids = ox.nearest_nodes(
        G,
        prov_valid["centroid_x"].values,
        prov_valid["centroid_y"].values
    )

    # Look up the projected coordinates of each snapped node.
    snapped_x = np.array([node_coord_dict[n][0] for n in snapped_node_ids])
    snapped_y = np.array([node_coord_dict[n][1] for n in snapped_node_ids])

    # Compute Euclidean distance between SAL centroid and snapped node, in meters.
    snap_dist = np.sqrt(
        (prov_valid["centroid_x"].values - snapped_x) ** 2 +
        (prov_valid["centroid_y"].values - snapped_y) ** 2
    )

    # Write snap distances back to the correct rows of all_access.
    all_access.loc[prov_valid.index, col_name] = snap_dist

    print(f"  {col_name}: mean={snap_dist.mean():.0f}m, "
          f"median={np.median(snap_dist):.0f}m, "
          f"max={snap_dist.max():.0f}m, "
          f"flagged >500m: {(snap_dist > 500).sum()}")

    # Free graph memory before loading the next one.
    del G, G_raw
    print()

print("SNAP DISTANCE COMPUTATION COMPLETE")

## DIAGNOSTIC SUMMARY

In [ ]:
# Summarize snap distance distributions across provinces and modes.
print("SNAP DISTANCE SUMMARY")
print()

for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0].capitalize()
    print(f"{mode} snap distances (meters):")
    for province in ["Gauteng", "KwaZulu-Natal"]:
        mask = all_access["PR_NAME"] == province
        vals = all_access.loc[mask, col].dropna()
        print(f"  {province}:")
        print(f"    mean={vals.mean():.0f}, median={np.median(vals):.0f}, "
              f"p95={np.percentile(vals, 95):.0f}, max={vals.max():.0f}")
        print(f"    SALs with snap > 100m: {(vals > 100).sum()} "
              f"({(vals > 100).mean() * 100:.1f}%)")
        print(f"    SALs with snap > 500m: {(vals > 500).sum()} "
              f"({(vals > 500).mean() * 100:.1f}%)")
        print(f"    SALs with snap > 1000m: {(vals > 1000).sum()} "
              f"({(vals > 1000).mean() * 100:.1f}%)")
    print()

In [ ]:
# Cross-tab of high snap distance by geography type.
# SALs with snap > 500m are worth flagging in analysis.
SNAP_FLAG_THRESHOLD_M = 500

pop_data = pd.read_csv(IN_POP_PRED)
pop_data["EA_CODE"] = pd.to_numeric(pop_data["EA_CODE"], errors = "coerce").astype("Int64")
pop_subset = pop_data[["EA_CODE", "EA_GTYPE", "EA_TYPE"]].copy()

# Drop any columns from all_access that overlap with pop_subset (excluding the merge key)
# to avoid pandas adding _x/_y suffixes.
overlap_cols = [c for c in ["EA_GTYPE", "EA_TYPE"] if c in all_access.columns]
diag = all_access.drop(columns = overlap_cols).merge(pop_subset, on = "EA_CODE", how = "left")

for col in ["walk_snap_dist_m", "drive_snap_dist_m"]:
    mode = col.split("_")[0]
    flag_col = f"{mode}_snap_flagged"
    diag[flag_col] = diag[col] > SNAP_FLAG_THRESHOLD_M

    print(f"{mode.upper()} SNAP FLAGS (>{SNAP_FLAG_THRESHOLD_M}m) BY GEOGRAPHY TYPE")
    print(
        diag.groupby(["PR_NAME", "EA_GTYPE"])[flag_col]
        .agg(["sum", "count", "mean"])
        .rename(columns = {"sum": "n_flagged", "count": "n_total", "mean": "pct_flagged"})
        .assign(pct_flagged = lambda x: (x["pct_flagged"] * 100).round(1))
        .to_string()
    )
    print()

## SAVE

In [ ]:
# Drop centroid helper columns before saving.
# These are already available in sal_pharmacy_distances_k3.csv.
all_access = all_access.drop(columns = ["centroid_x", "centroid_y"], errors = "ignore")

# Confirm final column additions.
print(f"NEW COLUMNS ADDED: walk_snap_dist_m, drive_snap_dist_m")
print(f"walk_snap_dist_m non-null: {all_access['walk_snap_dist_m'].notna().sum()}")
print(f"drive_snap_dist_m non-null: {all_access['drive_snap_dist_m'].notna().sum()}")
print(f"Total rows: {len(all_access)}")

# Overwrite all_access.
all_access.to_csv(OUT_ALL_ACCESS_SNAP, index = False)
print(f"SAVED: {OUT_ALL_ACCESS_SNAP}")